# 🌿 Plant Disease Detection — Full Pipeline
MobileNetV2 fine-tuned on PlantVillage (15 classes). Run cells top-to-bottom.

In [ ]:
# ── 1. Install dependencies ──────────────────────────────────────────────────
!pip install -q tensorflow==2.16.1 numpy==1.26.4 pillow==10.3.0 matplotlib scikit-learn

In [ ]:
# ── 2. Imports ────────────────────────────────────────────────────────────────
import os, io, zipfile, shutil, random
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from PIL import Image
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns

print('TF:', tf.__version__, '| NumPy:', np.__version__)
print('GPU:', tf.config.list_physical_devices('GPU') or 'None — using CPU')

In [ ]:
# ── 3a. Patch DepthwiseConv2D for Teachable Machine .h5 compatibility ─────────
_orig_dw = keras.layers.DepthwiseConv2D.__init__
def _patched_dw(self, *args, **kwargs):
    kwargs.pop('groups', None)
    _orig_dw(self, *args, **kwargs)
keras.layers.DepthwiseConv2D.__init__ = _patched_dw
print('DepthwiseConv2D patched ✅')

In [ ]:
# ── 3b. Upload existing model (optional — skip to Section 5 to train fresh) ───
from google.colab import files
print('Upload your model.h5 file (or skip and train from scratch in Section 5)')
uploaded = files.upload()  # upload model.h5
MODEL_PATH = list(uploaded.keys())[0] if uploaded else None
print('Uploaded:', MODEL_PATH)

In [ ]:
# ── 3c. Upload dataset zip ────────────────────────────────────────────────────
# Expected structure inside zip:
#   dataset/
#     ClassName1/  image1.jpg ...
#     ClassName2/  ...
print('Upload your dataset zip (folder-per-class structure)')
ds_upload = files.upload()
zip_name = list(ds_upload.keys())[0]
DATASET_DIR = '/content/dataset'
with zipfile.ZipFile(zip_name, 'r') as z:
    z.extractall(DATASET_DIR)
print('Classes found:', sorted(os.listdir(DATASET_DIR)))

In [ ]:
# ── 4. Data preprocessing ─────────────────────────────────────────────────────
IMG_SIZE   = (224, 224)
BATCH_SIZE = 32
SEED       = 42

train_ds = keras.utils.image_dataset_from_directory(
    DATASET_DIR,
    validation_split=0.2,
    subset='training',
    seed=SEED,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode='categorical',
)
val_ds = keras.utils.image_dataset_from_directory(
    DATASET_DIR,
    validation_split=0.2,
    subset='validation',
    seed=SEED,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode='categorical',
)

CLASS_LABELS = train_ds.class_names
NUM_CLASSES  = len(CLASS_LABELS)
print('Classes:', CLASS_LABELS)

# Normalize [0,255] → [0,1] and prefetch
normalize = layers.Rescaling(1.0 / 255)
train_ds  = train_ds.map(lambda x, y: (normalize(x), y), num_parallel_calls=tf.data.AUTOTUNE).prefetch(tf.data.AUTOTUNE)
val_ds    = val_ds.map(lambda x, y: (normalize(x), y),   num_parallel_calls=tf.data.AUTOTUNE).prefetch(tf.data.AUTOTUNE)

# Preview
plt.figure(figsize=(10, 4))
for images, labels in train_ds.take(1):
    for i in range(min(6, BATCH_SIZE)):
        plt.subplot(2, 3, i + 1)
        plt.imshow(images[i].numpy())
        plt.title(CLASS_LABELS[np.argmax(labels[i])])
        plt.axis('off')
plt.tight_layout()
plt.show()

In [ ]:
# ── 5. Model — load existing OR build + train from scratch ────────────────────
if MODEL_PATH and os.path.exists(MODEL_PATH):
    print('Loading uploaded model...')
    model = keras.models.load_model(MODEL_PATH, compile=False)
    print('Model loaded. Output shape:', model.output_shape)
else:
    print('Building MobileNetV2 transfer-learning model...')
    base = keras.applications.MobileNetV2(
        input_shape=(*IMG_SIZE, 3),
        include_top=False,
        weights='imagenet',
    )
    base.trainable = False

    inputs = keras.Input(shape=(*IMG_SIZE, 3))
    x = base(inputs, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(128, activation='relu')(x)
    x = layers.Dropout(0.3)(x)
    outputs = layers.Dense(NUM_CLASSES, activation='softmax')(x)
    model = keras.Model(inputs, outputs)

    model.compile(
        optimizer=keras.optimizers.Adam(1e-3),
        loss='categorical_crossentropy',
        metrics=['accuracy'],
    )
    model.summary()

    callbacks = [
        keras.callbacks.EarlyStopping(patience=5, restore_best_weights=True),
        keras.callbacks.ReduceLROnPlateau(factor=0.5, patience=3),
    ]
    history = model.fit(train_ds, validation_data=val_ds, epochs=20, callbacks=callbacks)

    # Fine-tune top layers
    base.trainable = True
    for layer in base.layers[:-30]:
        layer.trainable = False
    model.compile(
        optimizer=keras.optimizers.Adam(1e-5),
        loss='categorical_crossentropy',
        metrics=['accuracy'],
    )
    model.fit(train_ds, validation_data=val_ds, epochs=10, callbacks=callbacks)

In [ ]:
# ── 6. Evaluation ─────────────────────────────────────────────────────────────
loss, acc = model.evaluate(val_ds, verbose=0)
print(f'Validation loss: {loss:.4f} | Accuracy: {acc:.4f}')

y_true, y_pred = [], []
for images, labels in val_ds:
    preds = model.predict(images, verbose=0)
    y_true.extend(np.argmax(labels.numpy(), axis=1))
    y_pred.extend(np.argmax(preds, axis=1))

print('\nClassification Report:')
print(classification_report(y_true, y_pred, target_names=CLASS_LABELS))

cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(max(6, NUM_CLASSES), max(5, NUM_CLASSES - 1)))
sns.heatmap(cm, annot=True, fmt='d', xticklabels=CLASS_LABELS, yticklabels=CLASS_LABELS, cmap='Blues')
plt.ylabel('True'); plt.xlabel('Predicted'); plt.title('Confusion Matrix')
plt.tight_layout(); plt.show()

In [ ]:
# ── 7. Predict on sample images ───────────────────────────────────────────────
CONFIDENCE_THRESHOLD = 0.6

def predict_image(img_path):
    img = Image.open(img_path).convert('RGB').resize(IMG_SIZE)
    arr = np.expand_dims(np.array(img, dtype=np.float32) / 255.0, axis=0)
    preds = model.predict(arr, verbose=0)[0]
    confidence = float(np.max(preds))
    if confidence < CONFIDENCE_THRESHOLD:
        return 'Uncertain', confidence
    return CLASS_LABELS[int(np.argmax(preds))], confidence

# Collect up to 6 sample images from validation set
sample_paths = []
for cls in os.listdir(DATASET_DIR):
    cls_dir = os.path.join(DATASET_DIR, cls)
    if os.path.isdir(cls_dir):
        imgs = [os.path.join(cls_dir, f) for f in os.listdir(cls_dir) if f.lower().endswith(('.jpg','.jpeg','.png'))]
        if imgs:
            sample_paths.append(random.choice(imgs))
    if len(sample_paths) >= 6:
        break

plt.figure(figsize=(12, 4))
for i, path in enumerate(sample_paths):
    label, conf = predict_image(path)
    plt.subplot(2, 3, i + 1)
    plt.imshow(Image.open(path))
    plt.title(f'{label}\n{conf:.2%}', fontsize=9)
    plt.axis('off')
plt.tight_layout(); plt.show()

In [ ]:
# ── 8. Save model ─────────────────────────────────────────────────────────────
SAVE_PATH_H5    = '/content/model.h5'
SAVE_PATH_KERAS = '/content/model.keras'

model.save(SAVE_PATH_H5)
model.save(SAVE_PATH_KERAS)
print('Saved:', SAVE_PATH_H5)
print('Saved:', SAVE_PATH_KERAS)

In [ ]:
# ── 9. Download model ─────────────────────────────────────────────────────────
from google.colab import files
files.download(SAVE_PATH_H5)
# files.download(SAVE_PATH_KERAS)  # uncomment to also download .keras format